# <center> **MACHINE LEARNING**

### **1) IMPORTS LIBRARIES AND DATASETS**

#### **a) Libraries**

In [9]:
# general imports 
import os
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
import pickle

#### **b) Datasets**

In [10]:
ratings = pd.read_parquet("C://Users/barba/Case_studies/Cinema_recommender/cleaned_data/ratings.parquet", engine='pyarrow')
movies = pd.read_parquet("C://Users/barba/Case_studies/Cinema_recommender/cleaned_data/movies.parquet", engine='pyarrow')
genres = pd.read_parquet("C://Users/barba/Case_studies/Cinema_recommender/cleaned_data/genres.parquet", engine='pyarrow')
directors = pd.read_parquet("C://Users/barba/Case_studies/Cinema_recommender/cleaned_data/directors.parquet", engine='pyarrow')
actors = pd.read_parquet("C://Users/barba/Case_studies/Cinema_recommender/cleaned_data/actors.parquet", engine='pyarrow')

#### **c) Merging tables**

In [11]:
# Creation of a meta_table by merging useful tables: 
meta_table = movies.copy()
meta_table = meta_table.merge(ratings, how='left', on='tconst')
meta_table = meta_table.merge(directors, how='left', on='tconst')
meta_table = meta_table.merge(actors, how= 'left', on='tconst' )
meta_table = meta_table.merge(genres, how='left', on='tconst' )

#### **d) Final table**

In [12]:
# renaming columns 
meta_table = meta_table.rename(columns={'primaryName_x': 'director_name',
                                        'primaryName_y': 'actor_name',
                                        'startYear': 'year'})
# drop some columns that add noise such as nconst, birth and death year, region, categories 
meta_table= meta_table[['tconst','primaryTitle', 'title', 'director_name','actor_name', 'genres', 'year', 'detected_language','runtimeMinutes', 'averageRating', 'numVotes']]

In [13]:
# CLEANING 
meta_table.drop_duplicates(inplace=True) # suppression of more than 100000 duplicates
meta_table.dropna(inplace=True) # suppression of null values for our machine learning algorithm 

In [14]:
meta_table.sample(n=10, random_state=42) # this is a sample of 10 rows from our final meta_table 

,tconst,primaryTitle,title,director_name,actor_name,genres,year,detected_language,runtimeMinutes,averageRating,numVotes
594565,tt5153236,Hampstead,Hampstead,Joel Hopkins,Brendan Gleeson,Comedy,2017,en,102,6.1,8243.0
535180,tt3129692,Stealing Chanel,Le voleur au grand cœur,Roberto Mitrotti,Anna Maria Cianciulli,Romance,2015,en,102,5.4,94.0
260101,tt0365478,Man with the Screaming Brain,Man with the Screaming Brain,Bruce Campbell,Jonas Talkington,Comedy,2005,en,90,5.4,5810.0
417930,tt1558741,Comme les cinq doigts de la main,Comme les cinq doigts de la main,Alexandre Arcady,Mathieu Delarive,Drama,2010,fr,116,5.3,611.0
23463,tt0085501,Erendira,Erendira,Ruy Guerra,Irene Papas,Drama,1983,fr,103,6.6,730.0
614423,tt6034966,The Lion Sleeps Tonight,Le lion est mort ce soir,Nobuhiro Suwa,Pauline Etienne,Drama,2017,fr,103,6.4,330.0
424426,tt1610525,Chuck,Outsider,Philippe Falardeau,Naomi Watts,Biography,2016,en,98,6.5,7355.0
266346,tt0383010,The Three Stooges,Les trois corniauds,Peter Farrelly,Will Sasso,Comedy,2012,en,92,5.2,34406.0
615620,tt6105098,The Lion King,Le Roi lion,Jon Favreau,JD McCrary,Drama,2019,en,118,6.8,288298.0
606280,tt5751998,Small Town Crime,Small Town Crime,Eshom Nelms,Jeremy Ratchford,Action,2017,en,91,6.6,13206.0


### **2) FEATURE INGINEERING**

#### **a) High Value Features**

* **genres** : The #1 predictor. A user watching a "Comedy" wants another "Comedy."
* **director_names** (High Value): Directors carry a specific style/tone. 
* **actor_names** (High Value): For a general audience (families/seniors), "Star Power" matters. They might search "Jean Dujardin."
* **detected_language** (Specific to your Client): You should include this in the soup to group French movies together. This ensures a French movie is mathematically "closer" to other French movies than to Hollywood movies. 

#### **b) Filters for Ranking, Sorting**

* **averageRating**: To ensure you don't recommend bad movies (e.g., filter rating > 6.0).
* **numVotes**: To avoid recommending obscure movies with only 5 votes.
* **year** (startYear): To apply your "Nostalgia" logic (e.g., boost movies between 1980–2000).
* **runtimeMinutes**: Optional, but good for the user interface (e.g., "Short movies under 90 min").

#### **c) Features Extraction**

In [15]:
# Select only the features relevant for the Recommender System
features_df = meta_table[[
    'tconst',            # Keep ID to track the movie
    'primaryTitle',      # International Title (for search)
    'title',             # French Title (for display)
    'genres',            # SOUP INGREDIENT
    'director_name',    # SOUP INGREDIENT
    'actor_name',       # SOUP INGREDIENT
    'detected_language', # SOUP INGREDIENT
    'averageRating',     # FILTER
    'numVotes',          # FILTER
    'year',
    'runtimeMinutes'# FILTER
]].copy()

### **3) VECTORIZATION**

* Goal: Turn your 20,000+ movie "soups" into a giant matrix of numbers.
* Tool: CountVectorizer (better for keywords/names) or TfidfVectorizer (better for long plot descriptions). Since you are using metadata (Actors, Genres), use CountVectorizer.
* Optimization: Use stop_words='english' to remove noise, even though your content is French/International, names don't have stop words.

In [16]:
# CREATE THE "SOUP" (Feature Engineering) 
# # We mix all text features into one long string.
# We weight features by repeating them (Strategy from Presentation)
def create_soup(x):
    return (
        (str(x['actor_name']) + ' ') * 3 +  # Weight: 3x (Strongest Driver)
        (str(x['genres']) + ' ') * 2 +          # Weight: 2x (Habit Driver)
        (str(x['detected_language']) + ' ') * 2 + # Weight: 2x (French Preference)
        (str(x['director_name']) + ' ') * 2  +
        str(x['primaryTitle']) # Helping search accuracy
    ).lower()

features_df['soup'] = features_df.apply(create_soup, axis=1)

In [17]:
features_df

,tconst,primaryTitle,title,genres,director_name,actor_name,detected_language,averageRating,numVotes,year,runtimeMinutes,soup
0,tt0035423,Kate & Leopold,Kate et Léopold,Comedy,James Mangold,Meg Ryan,en,6.4,93042.0,2001,118,meg ryan meg ryan meg ryan comedy comedy en en...
1,tt0035423,Kate & Leopold,Kate et Léopold,Fantasy,James Mangold,Meg Ryan,en,6.4,93042.0,2001,118,meg ryan meg ryan meg ryan fantasy fantasy en ...
2,tt0035423,Kate & Leopold,Kate et Léopold,Romance,James Mangold,Meg Ryan,en,6.4,93042.0,2001,118,meg ryan meg ryan meg ryan romance romance en ...
3,tt0035423,Kate & Leopold,Kate et Léopold,Comedy,James Mangold,Hugh Jackman,en,6.4,93042.0,2001,118,hugh jackman hugh jackman hugh jackman comedy ...
4,tt0035423,Kate & Leopold,Kate et Léopold,Fantasy,James Mangold,Hugh Jackman,en,6.4,93042.0,2001,118,hugh jackman hugh jackman hugh jackman fantasy...
...,...,...,...,...,...,...,...,...,...,...,...,...
677527,tt9908390,Le lion,Le lion,Comedy,Ludovic Colbeau-Justin,Carole Brana,fr,5.5,1601.0,2020,95,carole brana carole brana carole brana comedy ...
677528,tt9908390,Le lion,Le lion,Comedy,Ludovic Colbeau-Justin,Nicolas Briançon,fr,5.5,1601.0,2020,95,nicolas briançon nicolas briançon nicolas bria...
677529,tt9908390,Le lion,Le lion,Comedy,Ludovic Colbeau-Justin,Ophélia Kolb,fr,5.5,1601.0,2020,95,ophélia kolb ophélia kolb ophélia kolb comedy ...
677530,tt9908390,Le lion,Le lion,Comedy,Ludovic Colbeau-Justin,Philippe Duquesne,fr,5.5,1601.0,2020,95,philippe duquesne philippe duquesne philippe d...


In [18]:
# VECTORIZATION (Text to Numbers)
# We use CountVectorizer because we care about frequency of specific tags (Genre, Name).
# stop_words='english' removes "the", "and", etc.
count = CountVectorizer(stop_words='english', min_df=1)
count_matrix = count.fit_transform(features_df['soup'])
print(f"🔢 Matrix Shape: {count_matrix.shape} (Movies, Unique Words)")

🔢 Matrix Shape: (497225, 77446) (Movies, Unique Words)


### **4) MODEL: NearestNeighbors**

In [19]:
# TRAIN THE MODEL
# We use 'cosine' metric so it behaves exactly like cosine_similarity
# algorithm='brute' is actually often fastest for sparse matrices, but 'auto' is fine.
model_knn = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=20, n_jobs=-1)

# This "fit" doesn't compute the big matrix. It just indexes the data. (Fast & Low Memory)
model_knn.fit(count_matrix)

print("✅ Model Trained successfully")

✅ Model Trained successfully


In [21]:
# EXPORT
# Save the trained model and the matrix
out_dir = r"C:\Users\barba\Case_studies\Cinema_recommender\app"
with open(os.path.join(out_dir, 'knn_model.pkl'), 'wb') as f:
	pickle.dump(model_knn, f)
with open(os.path.join(out_dir, 'count_matrix.pkl'), 'wb') as f:
	pickle.dump(count_matrix, f)
with open(os.path.join(out_dir, 'movie_list.pkl'), 'wb') as f:
	pickle.dump(features_df, f)

print("💾 Files saved for Streamlit!")

💾 Files saved for Streamlit!
